# Week 02 - Lesson 06: Multi-Agent Systems: Swarm

## Overview

This notebook introduces **Swarm Multi-Agent Systems**—dynamic AI architectures where specialized agents collaborate through intelligent handoffs to solve complex business problems. You'll learn how to design and implement swarm systems using LangGraph's handoff mechanisms.

In this material we will focus on the **Swarm Architecture** pattern, where agents dynamically transfer control to one another based on their specializations and the current context. You'll see how this pattern enables sophisticated business process automation and customer service workflows.

### Learning Objectives
By the end of this notebook, you will:
1. Master swarm architecture patterns and dynamic agent handoffs
2. Implement intelligent agent communication and state management
3. Build production-ready customer service swarm systems
4. Understand advanced multi-agent orchestration techniques

Swarm systems represent the pinnacle of multi-agent AI, enabling:

- **Dynamic Specialization**: Agents automatically route tasks to the most qualified specialist
- **Intelligent Handoffs**: Seamless transfer of control based on context and expertise
- **Scalable Collaboration**: Systems that grow and adapt as new agents are added
- **Business Process Automation**: End-to-end workflows with intelligent decision points

---

## Environment Setup and Dependencies

First, let's ensure we have all necessary dependencies installed and configured for our swarm multi-agent system.


In [1]:
# Install required dependencies for swarm multi-agent systems
%pip install -U --quiet langchain langchain-openai langgraph langgraph-swarm python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
# Configure API Keys
import os
os.environ["OPENAI_API_KEY"] = "sk-your-openai-api-key"

In [ ]:
# Import necessary libraries for swarm multi-agent systems
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool

# LangGraph imports
from langgraph.graph.message import add_messages

# LangGraph Swarm imports
from langgraph.prebuilt import create_react_agent
from langgraph_swarm import create_swarm, create_handoff_tool

# Load environment variables
load_dotenv()

print("✅ All swarm multi-agent dependencies imported successfully!")


✅ All swarm multi-agent dependencies imported successfully!


## Setting Up the Foundation

Let's initialize our core models and define the shared state structure for our swarm multi-agent system.


In [4]:
# LangChain core imports
from langchain_openai import ChatOpenAI

# Initialize the language model for all agents
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0.1,
)

print(f"✅ Model initialized: {llm.model_name}")
print(f"Temperature: {llm.temperature}")

✅ Model initialized: gpt-4o-mini
Temperature: 0.1


---

## Swarm Architecture:

In swarm-based multi-agent systems, agents dynamically hand off control to one another based on their specializations. This architecture is ideal for complex business processes that require multiple specialized agents working together.

### Swarm Architecture Characteristics

- **Dynamic Routing**: Agents decide which specialist to hand off to based on context
- **Specialized Expertise**: Each agent focuses on specific business domains
- **Intelligent Handoffs**: Seamless transfer of control with context preservation

---

#### **Example: Customer Service Swarm**

Let's implement a customer service system with four specialized agents:
1. **Reception Agent**: Initial customer contact and issue classification
2. **Technical Agent**: Handles technical support and troubleshooting
3. **Billing Agent**: Manages billing inquiries and payment issues
4. **Escalation Agent**: Handles complex issues requiring human intervention


In [5]:
# Define tools for customer service agents
@tool
def check_account_status(customer_id: str) -> str:
    """Check customer account status and subscription details"""
    # In real-world scenarios, this tool would fetch the actual account status from a database or API
    return f"Account Status: Active | Subscription: Premium | Last Payment: 2024-01-15 | Customer ID: {customer_id} "

@tool
def troubleshoot_technical_issue(issue_description: str) -> str:
    """Troubleshoot technical issues and provide solutions"""
    # In real-world scenarios, this tool would perform a detailed analysis of the issue and provide a solution
    return f"Technical Analysis: {issue_description} | Solution: Clear cache and restart application | Status: Resolved"

@tool
def process_billing_inquiry(inquiry_type: str) -> str:
    """Process billing inquiries and payment issues and provide a resolution"""
    # In real-world scenarios, this tool would process the billing inquiry and provide a resolution
    return f"Billing Inquiry: {inquiry_type} | Resolution: Payment processed successfully | Next billing: 2024-02-15"

@tool
def escalate_to_human(issue_complexity: str) -> str:
    """Escalate complex issues to human support team and return a escalation message"""
    # In real-world scenarios, this tool would escalate the issue to a human support team and provide an ETA
    return f"Escalation: {issue_complexity} | Priority: High | Assigned to: Senior Support Team | ETA: 2 hours"

print("✅ Customer service tools created!")

✅ Customer service tools created!


In [6]:
# Create handoff tools for agent communication
transfer_to_technical = create_handoff_tool(
    agent_name="technical_agent",
    description="Transfer customer to technical support specialist for technical issues."
)

transfer_to_billing = create_handoff_tool(
    agent_name="billing_agent",
    description="Transfer customer to billing specialist for payment and subscription issues."
)

transfer_to_escalation = create_handoff_tool(
    agent_name="escalation_agent",
    description="Transfer customer to escalation specialist for complex issues."
)

transfer_to_reception = create_handoff_tool(
    agent_name="reception_agent",
    description="Transfer customer back to reception for general assistance."
)

print("✅ Handoff tools created!")


✅ Handoff tools created!


In [7]:
# Create specialized customer service agents
reception_agent = create_react_agent(
    model=llm,
    tools=[check_account_status, transfer_to_technical, transfer_to_billing, transfer_to_escalation],
    prompt="""You are a customer service reception agent. Your role is to:
    1. Greet customers and understand their issues
    2. Check account status using available tools
    3. Route customers to appropriate specialists:
       - Technical issues → technical_agent
       - Billing/payment issues → billing_agent
       - Complex issues requiring human intervention → escalation_agent
    4. Always be professional and helpful
    Use the handoff tools to transfer customers to the right specialist.""",
    name="reception_agent"
)

technical_agent = create_react_agent(
    model=llm,
    tools=[troubleshoot_technical_issue, transfer_to_reception, transfer_to_escalation],
    prompt="""You are a technical agent that can use the troubleshoot technical issue tool to provide solutions about technical issues. Your role is to:
    1. Use troubleshooting technical issue tool to provide solutions about technical issues
    2. Transfer back to reception if issue is resolved
    3. Transfer to escalation if issue is complex
    4. Transfer back to reception for follow-up""",
    name="technical_agent"
)

billing_agent = create_react_agent(
    model=llm,
    tools=[process_billing_inquiry, transfer_to_reception, transfer_to_escalation],
    prompt="""You are a billing agent that can use the process billing inquiry tool to provide solutions about billing issues. Your role is to:
    1. Use process billing inquiry tool to provide solutions about billing issues
    3. Transfer to escalation if issue is complex
    4. Transfer back to reception for follow-up

    Focus on resolving billing concerns efficiently.""",
    name="billing_agent"
)

escalation_agent = create_react_agent(
    model=llm,
    tools=[escalate_to_human, transfer_to_reception],
    prompt="""You are an escalation agent that can use the escalate to human tool to provide solutions about complex issues. Your role is to:
    1. Use escalate to human tool to provide solutions about complex issues
    2. Transfer back to reception if issue is resolved
    3. Provide customers with clear next steps and timelines
    4. Transfer back to reception for follow-up
    
    Focus on ensuring complex issues are properly escalated.""",
    name="escalation_agent"
)

print("✅ Specialized customer service agents created!")


✅ Specialized customer service agents created!


In [8]:
# Create the swarm system
customer_service_swarm = create_swarm(
    agents=[reception_agent, technical_agent, billing_agent, escalation_agent],
    default_active_agent="reception_agent"
).compile()

print("✅ Customer service swarm system created!")
print("Available agents: reception_agent, technical_agent, billing_agent, escalation_agent")
print("Default active agent: reception_agent")


✅ Customer service swarm system created!
Available agents: reception_agent, technical_agent, billing_agent, escalation_agent
Default active agent: reception_agent


## Testing the Swarm System

Let's test our customer service swarm with various real-world scenarios to demonstrate how agents dynamically hand off control to one another.


In [9]:
def run_swarm_messages(scenario_name: str, customer_message: str):
    """Clean test focusing on agent responses."""
    print(f"\n🔍 {scenario_name}")
    print(f"Customer: {customer_message}")
    print("=" * 80)
    
    try:
        for chunk in customer_service_swarm.stream(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": customer_message
                    }
                ]
            }
        ):
            if chunk:
                for agent_name, agent_response in chunk.items():
                    if 'messages' in agent_response:
                        print(f"\n🤖 {agent_name.upper()}:")
                        for message in agent_response['messages']:
                            if hasattr(message, 'content') and message.content:
                                if hasattr(message, 'role'):
                                    if message.role == 'tool':
                                        print(f"  🔧 Tool: {message.content}")
                                    else:
                                        print(f"  💬 {message.content}")
                                else:
                                    print(f"  💬 {message.content}")
                        print("-" * 60)
    
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

In [10]:
# Test scenario
run_swarm_messages(
    "Technical Support", 
    "I can't login to my account. check my status. my ID is 1234567890"
)


🔍 Technical Support
Customer: I can't login to my account. check my status. my ID is 1234567890

🤖 RECEPTION_AGENT:
  💬 I can't login to my account. check my status. my ID is 1234567890
  💬 Account Status: Active | Subscription: Premium | Last Payment: 2024-01-15 | Customer ID: 1234567890 
  💬 Your account status is active, and you have a premium subscription. The last payment was made on January 15, 2024. 

Since you're having trouble logging in, I will transfer you to our technical support specialist for assistance. Please hold on for a moment.
  💬 Successfully transferred to technical_agent
------------------------------------------------------------

🤖 TECHNICAL_AGENT:
  💬 I can't login to my account. check my status. my ID is 1234567890
  💬 Account Status: Active | Subscription: Premium | Last Payment: 2024-01-15 | Customer ID: 1234567890 
  💬 Your account status is active, and you have a premium subscription. The last payment was made on January 15, 2024. 

Since you're having t

In [11]:
run_swarm_messages(
    "Billing Issue", 
    "I was charged twice this month. Need a refund. My ID is 1234567890"
)


🔍 Billing Issue
Customer: I was charged twice this month. Need a refund. My ID is 1234567890

🤖 RECEPTION_AGENT:
  💬 I was charged twice this month. Need a refund. My ID is 1234567890
  💬 Account Status: Active | Subscription: Premium | Last Payment: 2024-01-15 | Customer ID: 1234567890 
  💬 Successfully transferred to billing_agent
------------------------------------------------------------

🤖 BILLING_AGENT:
  💬 I was charged twice this month. Need a refund. My ID is 1234567890
  💬 Account Status: Active | Subscription: Premium | Last Payment: 2024-01-15 | Customer ID: 1234567890 
  💬 Successfully transferred to billing_agent
  💬 Billing Inquiry: double charge refund request | Resolution: Payment processed successfully | Next billing: 2024-02-15
  💬 Your double charge inquiry has been processed successfully. A refund for the duplicate charge will be initiated, and your next billing date is set for February 15, 2024. If you have any further questions or need assistance, please let me

In [12]:

run_swarm_messages(
    "Complex Issue", 
    "I need to cancel my account but have a custom integration. This is urgent."
)



🔍 Complex Issue
Customer: I need to cancel my account but have a custom integration. This is urgent.

🤖 RECEPTION_AGENT:
  💬 I need to cancel my account but have a custom integration. This is urgent.
  💬 I understand that you need to cancel your account and that you have a custom integration, which makes this a complex issue. I will escalate this matter to a specialist who can assist you further. Please hold on for a moment. 

Transferring you now.
  💬 Successfully transferred to escalation_agent
------------------------------------------------------------

🤖 ESCALATION_AGENT:
  💬 I need to cancel my account but have a custom integration. This is urgent.
  💬 I understand that you need to cancel your account and that you have a custom integration, which makes this a complex issue. I will escalate this matter to a specialist who can assist you further. Please hold on for a moment. 

Transferring you now.
  💬 Successfully transferred to escalation_agent
  💬 Escalation: Account cancellati

## Key Takeaways

### What We've Accomplished

- **Dynamic Agent Routing**: Agents that intelligently hand off control based on context
- **State Management**: Preserving context and information across agent transitions
- **Handoff Mechanisms**: Implementing seamless agent-to-agent communication